# Chapitre 5 - Cours 2 : splines de lissage, degrés de liberté et choix de $\lambda$

Les splines de régression du cours 1 obligent à choisir le nombre et la position des
nœuds. Les **splines de lissage** contournent ce problème : on met un nœud en **chaque**
observation et l'on contrôle la complexité par une **pénalité** de courbure. Plan :

1. le critère pénalisé et sa solution (§5.4) ;
2. calcul matriciel en base de B-splines, données *bone* (figure 5.6) ;
3. matrice de lissage, degrés de liberté effectifs, forme de Reinsch et valeurs
   propres (§5.4.1, figures 5.7 et 5.8) ;
4. choix de $\lambda$ : compromis biais-variance et validation croisée (§5.5, figure 5.9) ;
5. régression logistique non paramétrique (§5.6).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import datasets
from esl.ch05_splines import (SmoothingSpline, bspline_basis, bspline_penalty_matrix,
                              natural_spline_basis, loocv_score, gcv_score)
set_style()

## 1. Le critère des splines de lissage

Parmi toutes les fonctions $f$ deux fois dérivables, on cherche celle qui minimise la
**somme des carrés résiduels pénalisée** (éq. (5.9) du livre)

$$
\mathrm{RSS}(f, \lambda) = \sum_{i=1}^{N}\{y_i - f(x_i)\}^2
  + \lambda \int \{f''(t)\}^2\,dt .
$$

Le premier terme mesure l'**attache aux données**, le second la **courbure** de $f$
($f''$ est nulle pour une droite). Le paramètre $\lambda \ge 0$ règle le compromis :

- $\lambda = 0$ : aucune pénalité, $f$ peut être n'importe quelle fonction interpolant
  les données ;
- $\lambda = \infty$ : aucune courbure tolérée, $f$ est la droite des moindres carrés.

**Théorème (exercice 5.7).** Le problème, posé sur un espace de fonctions de dimension
infinie, admet une solution **unique**, qui est une **spline cubique naturelle** dont les
nœuds sont les valeurs distinctes des $x_i$.

> **Intuition.** Fixons les valeurs $f(x_i) = z_i$. Le premier terme ne dépend plus que
> des $z_i$ ; il reste à minimiser $\int f''^2$ parmi les fonctions qui passent par les
> points $(x_i, z_i)$ : c'est la spline cubique naturelle interpolante qui réalise ce
> minimum (exercice 5.7). On peut donc se restreindre à cet espace de dimension $N$.

## 2. Forme matricielle

Écrivons $f(x) = \sum_{j} N_j(x)\theta_j$ dans une base de cet espace. Avec
$\{\mathbf N\}_{ij} = N_j(x_i)$ et $\{\Omega_N\}_{jk} = \int N_j''(t)N_k''(t)\,dt$, on a
$\int f''^2 = \theta^T\Omega_N\theta$ (par bilinéarité de l'intégrale), donc (éq. (5.11))

$$
\mathrm{RSS}(\theta, \lambda) = (\mathbf y - \mathbf N\theta)^T(\mathbf y - \mathbf N\theta)
  + \lambda\,\theta^T\Omega_N\theta .
$$

Le gradient en $\theta$ vaut $-2\mathbf N^T(\mathbf y - \mathbf N\theta) + 2\lambda\Omega_N\theta$ ;
l'annuler donne (éq. (5.12))

$$
\hat\theta = (\mathbf N^T\mathbf N + \lambda\Omega_N)^{-1}\mathbf N^T\mathbf y ,
$$

une **régression ridge généralisée**. En pratique (annexe du chapitre, éq. (5.79)), on
travaille dans une base de **B-splines cubiques** : la pénalité impose d'elle-même la
linéarité aux bords, et les matrices sont creuses.

> **Remarque (implémentation).** La classe `SmoothingSpline` de `esl.ch05_splines` place
> les nœuds intérieurs aux valeurs distinctes $x_{(2)}, \dots, x_{(n-1)}$ et les nœuds de
> bord en $x_{(1)}$ et $x_{(n)}$ : c'est une base de $n + 2$ B-splines (exercice 5.8).
> La matrice $\Omega_B$ est calculée **exactement** : $B_j''$ est affine sur chaque
> intervalle entre nœuds, donc $B_j''B_k''$ est un polynôme de degré 2, intégré exactement
> par une quadrature de Gauss à 2 points.

Vérifions sur un exemple que $\gamma^T\Omega_B\gamma = \int f''^2$ pour
$f = \sum_j \gamma_j B_j$, en comparant à une intégration numérique fine.

In [2]:
rng = np.random.RandomState(0)
noeuds = np.sort(rng.uniform(0, 1, 8))
Om = bspline_penalty_matrix(noeuds, 0.0, 1.0)
gam = rng.randn(Om.shape[0])
tt = np.linspace(0, 1, 200001)
f2 = bspline_basis(tt, noeuds, 0.0, 1.0, deriv=2) @ gam
print("gamma^T Omega gamma  =", gam @ Om @ gam)
print("intégrale numérique  =", np.trapezoid(f2 ** 2, tt))

gamma^T Omega gamma  = 32731.739399694183
intégrale numérique  = 32731.739272045375


Le noyau de $\Omega_B$ (vecteurs $\gamma$ tels que $\gamma^T\Omega_B\gamma = 0$) correspond
aux fonctions de dérivée seconde nulle, c'est-à-dire aux **fonctions affines** : il est de
dimension 2.

In [3]:
vp = np.linalg.eigvalsh(Om)
print("plus petites valeurs propres de Omega :", np.round(vp[:4], 6) + 0.0)

plus petites valeurs propres de Omega : [  0.         0.        27.179188 211.390231]
